Name: Bryan Sandor

Course: STAT 558

Title: Project 2

# Preamble

The following code may be used to reload a module.

In [1]:
# importlib.reload(Name_of_module)

# Part I - Creating a Class

First we import all the necessary libraries and initiate a spark session.

In [2]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import importlib # used with importlib.reload(Name_of_Module) to reload module

from pyspark.sql import SparkSession
spark = SparkSession.builder \
        .appName("proj2") \
        .config("spark.sql.ansi.enabled", "false") \
        .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/27 01:21:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


A `python` file containing several methods on two classes was created. The two classes depend on whether the `spark` data frame was read in from a `csv` file or "converted" to a `spark` data frame from an existing `pandas` data frame. Thus, the two classes are `read_spark` and `convert_sdf`, respectively. The file is imported, next.

In [3]:
import proj2class

## Read-In Class

We focus on a file read in from a `csv` file and converted to a `spark` data frame, first:

In [4]:
classSparkAirDat = proj2class.SparkDataCheck.read_spark(spark, "AirQualityUCI.csv", delimiter = ';')
classSparkAirDat.df.show(10)

+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+
|      Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|_c15|_c16|
+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+
|10/03/2004|18.00.00|   2,6|       1360|     150|    11,9|         1046|    166|        1056|    113|        1692|       1268|13,6|48,9|0,7578|NULL|NULL|
|10/03/2004|19.00.00|     2|       1292|     112|     9,4|          955|    103|        1174|     92|        1559|        972|13,3|47,7|0,7255|NULL|NULL|
|10/03/2004|20.00.00|   2,2|       1402|      88|     9,0|          939|    131|        1140|    114|        1555|       1074|11,9|54,0|0,7502|NULL|NULL|
|10/03/2004|21.00.00|   2,2|       1376|      80|     9,2|          948|    

26/03/27 01:21:06 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, , 
 Schema: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, _c15, _c16
Expected: _c15 but found: 
CSV file: file:///home/jupyter-bgsandor@ncsu.edu/AirQualityUCI.csv


We observe a total of $16$ columns, with the last two being `NULL` type. Several of our methods will depend on the type of data in each column, so next we examine them.

In [5]:
classSparkAirDat.df.dtypes

[('Date', 'string'),
 ('Time', 'string'),
 ('CO(GT)', 'string'),
 ('PT08.S1(CO)', 'int'),
 ('NMHC(GT)', 'int'),
 ('C6H6(GT)', 'string'),
 ('PT08.S2(NMHC)', 'int'),
 ('NOx(GT)', 'int'),
 ('PT08.S3(NOx)', 'int'),
 ('NO2(GT)', 'int'),
 ('PT08.S4(NO2)', 'int'),
 ('PT08.S5(O3)', 'int'),
 ('T', 'string'),
 ('RH', 'string'),
 ('AH', 'string'),
 ('_c15', 'string'),
 ('_c16', 'string')]

### Method 1

The first method we demonstrate returns a data frame with an additional `boolean` column for whether data in a specific *numeric* column fall between two bounds, say $1000$ and $1250$ for the column `PT08.S1(CO)`.

In [6]:
classSparkAirDat.numrange("`PT08.S1(CO)`", 1000, 1250).show()

+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|      Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|_c15|_c16|Result|
+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|10/03/2004|18.00.00|   2,6|       1360|     150|    11,9|         1046|    166|        1056|    113|        1692|       1268|13,6|48,9|0,7578|NULL|NULL| false|
|10/03/2004|19.00.00|     2|       1292|     112|     9,4|          955|    103|        1174|     92|        1559|        972|13,3|47,7|0,7255|NULL|NULL| false|
|10/03/2004|20.00.00|   2,2|       1402|      88|     9,0|          939|    131|        1140|    114|        1555|       1074|11,9|54,0|0,7502|NULL|NULL| false|
|10/03/2004|21.00.00|   2,2|      

26/03/27 01:21:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, , 
 Schema: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, _c15, _c16
Expected: _c15 but found: 
CSV file: file:///home/jupyter-bgsandor@ncsu.edu/AirQualityUCI.csv


Next we observe if the lower bound is omitted, then $-\infty$ is assumed as a lower bound.

In [7]:
classSparkAirDat.numrange("`NO2(GT)`", upper = 100).show()

26/03/27 01:21:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, , 
 Schema: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, _c15, _c16
Expected: _c15 but found: 
CSV file: file:///home/jupyter-bgsandor@ncsu.edu/AirQualityUCI.csv


+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|      Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|_c15|_c16|Result|
+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|10/03/2004|18.00.00|   2,6|       1360|     150|    11,9|         1046|    166|        1056|    113|        1692|       1268|13,6|48,9|0,7578|NULL|NULL| false|
|10/03/2004|19.00.00|     2|       1292|     112|     9,4|          955|    103|        1174|     92|        1559|        972|13,3|47,7|0,7255|NULL|NULL|  true|
|10/03/2004|20.00.00|   2,2|       1402|      88|     9,0|          939|    131|        1140|    114|        1555|       1074|11,9|54,0|0,7502|NULL|NULL| false|
|10/03/2004|21.00.00|   2,2|      

A similar observation can be made if the upper bound is omitted, then $+\infty$ is assumed.

In [8]:
classSparkAirDat.numrange("`NMHC(GT)`", 30).show()

+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|      Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|_c15|_c16|Result|
+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|10/03/2004|18.00.00|   2,6|       1360|     150|    11,9|         1046|    166|        1056|    113|        1692|       1268|13,6|48,9|0,7578|NULL|NULL|  true|
|10/03/2004|19.00.00|     2|       1292|     112|     9,4|          955|    103|        1174|     92|        1559|        972|13,3|47,7|0,7255|NULL|NULL|  true|
|10/03/2004|20.00.00|   2,2|       1402|      88|     9,0|          939|    131|        1140|    114|        1555|       1074|11,9|54,0|0,7502|NULL|NULL|  true|
|10/03/2004|21.00.00|   2,2|      

26/03/27 01:21:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, , 
 Schema: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, _c15, _c16
Expected: _c15 but found: 
CSV file: file:///home/jupyter-bgsandor@ncsu.edu/AirQualityUCI.csv


Note that if both bounds are omitted, then all real values are considered, returning `true` for any real number in the data.

In [9]:
classSparkAirDat.numrange("`NOx(GT)`").show()

+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|      Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|_c15|_c16|Result|
+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|10/03/2004|18.00.00|   2,6|       1360|     150|    11,9|         1046|    166|        1056|    113|        1692|       1268|13,6|48,9|0,7578|NULL|NULL|  true|
|10/03/2004|19.00.00|     2|       1292|     112|     9,4|          955|    103|        1174|     92|        1559|        972|13,3|47,7|0,7255|NULL|NULL|  true|
|10/03/2004|20.00.00|   2,2|       1402|      88|     9,0|          939|    131|        1140|    114|        1555|       1074|11,9|54,0|0,7502|NULL|NULL|  true|
|10/03/2004|21.00.00|   2,2|      

26/03/27 01:21:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, , 
 Schema: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, _c15, _c16
Expected: _c15 but found: 
CSV file: file:///home/jupyter-bgsandor@ncsu.edu/AirQualityUCI.csv


Now we observe the consequence of attempting to submit a non-numeric column for the method.

In [10]:
classSparkAirDat.numrange("`CO(GT)`")

The column must be numeric!


### Method 2

Now we examine another method that focuses on strings, comparing strings in appropriate columns to strings given by the user. First notice it won't accept numeric columns

In [11]:
classSparkAirDat.strrange("`NOx(GT)`", "166")

The column must contain strings!


The `boolean` column appended to the end follows the same idea as the previous method.

In [12]:
classSparkAirDat.strrange("`CO(GT)`", "2,2").show()

+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|      Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|_c15|_c16|Result|
+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|10/03/2004|18.00.00|   2,6|       1360|     150|    11,9|         1046|    166|        1056|    113|        1692|       1268|13,6|48,9|0,7578|NULL|NULL| false|
|10/03/2004|19.00.00|     2|       1292|     112|     9,4|          955|    103|        1174|     92|        1559|        972|13,3|47,7|0,7255|NULL|NULL| false|
|10/03/2004|20.00.00|   2,2|       1402|      88|     9,0|          939|    131|        1140|    114|        1555|       1074|11,9|54,0|0,7502|NULL|NULL|  true|
|10/03/2004|21.00.00|   2,2|      

26/03/27 01:21:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, , 
 Schema: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, _c15, _c16
Expected: _c15 but found: 
CSV file: file:///home/jupyter-bgsandor@ncsu.edu/AirQualityUCI.csv


Note that the method does include partial strings within the column, hence the following query returns `true` whenever the column contains a $3$.

In [13]:
classSparkAirDat.strrange("`T`", "3").show()

+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|      Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|_c15|_c16|Result|
+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|10/03/2004|18.00.00|   2,6|       1360|     150|    11,9|         1046|    166|        1056|    113|        1692|       1268|13,6|48,9|0,7578|NULL|NULL|  true|
|10/03/2004|19.00.00|     2|       1292|     112|     9,4|          955|    103|        1174|     92|        1559|        972|13,3|47,7|0,7255|NULL|NULL|  true|
|10/03/2004|20.00.00|   2,2|       1402|      88|     9,0|          939|    131|        1140|    114|        1555|       1074|11,9|54,0|0,7502|NULL|NULL| false|
|10/03/2004|21.00.00|   2,2|      

26/03/27 01:21:07 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, , 
 Schema: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, _c15, _c16
Expected: _c15 but found: 
CSV file: file:///home/jupyter-bgsandor@ncsu.edu/AirQualityUCI.csv


### Method 3

The next method checks whether data in a column is "missing" or marked `NULL` or `NaN`.

In [14]:
classSparkAirDat.nulrange("`_c15`").show()

+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|      Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|_c15|_c16|Result|
+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|10/03/2004|18.00.00|   2,6|       1360|     150|    11,9|         1046|    166|        1056|    113|        1692|       1268|13,6|48,9|0,7578|NULL|NULL|  true|
|10/03/2004|19.00.00|     2|       1292|     112|     9,4|          955|    103|        1174|     92|        1559|        972|13,3|47,7|0,7255|NULL|NULL|  true|
|10/03/2004|20.00.00|   2,2|       1402|      88|     9,0|          939|    131|        1140|    114|        1555|       1074|11,9|54,0|0,7502|NULL|NULL|  true|
|10/03/2004|21.00.00|   2,2|      

26/03/27 01:21:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, , 
 Schema: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, _c15, _c16
Expected: _c15 but found: 
CSV file: file:///home/jupyter-bgsandor@ncsu.edu/AirQualityUCI.csv


versus

In [15]:
classSparkAirDat.nulrange("`T`").show()

+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|      Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|_c15|_c16|Result|
+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+----+----+------+
|10/03/2004|18.00.00|   2,6|       1360|     150|    11,9|         1046|    166|        1056|    113|        1692|       1268|13,6|48,9|0,7578|NULL|NULL| false|
|10/03/2004|19.00.00|     2|       1292|     112|     9,4|          955|    103|        1174|     92|        1559|        972|13,3|47,7|0,7255|NULL|NULL| false|
|10/03/2004|20.00.00|   2,2|       1402|      88|     9,0|          939|    131|        1140|    114|        1555|       1074|11,9|54,0|0,7502|NULL|NULL| false|
|10/03/2004|21.00.00|   2,2|      

26/03/27 01:21:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, , 
 Schema: Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH, _c15, _c16
Expected: _c15 but found: 
CSV file: file:///home/jupyter-bgsandor@ncsu.edu/AirQualityUCI.csv


### Method 4

Our next method computes the `min` and `max` of a *numeric* column, possibly grouped by an additional column. The output is a `pandas` data frame.

In [16]:
classSparkAirDat.minmax("`CO(GT)`")

The column must be numeric!


In [17]:
classSparkAirDat.minmax("`PT08.S1(CO)`")

,min(PT08.S1(CO)),max(PT08.S1(CO))
0,-200,2040


Finally, we group the data with two different columns
- `Date` (`string` type)
- `PT08.S2(NMHC)` (`int` type)

In [18]:
classSparkAirDat.minmax("`PT08.S1(CO)`", groupby = "`Date`")

,Date,min(PT08.S1(CO)),max(PT08.S1(CO))
0,23/07/2004,912.0,1367.0
1,28/07/2004,742.0,1210.0
2,10/10/2004,941.0,1520.0
3,10/11/2004,715.0,1277.0
4,16/12/2004,-200.0,-200.0
...,...,...,...
387,13/10/2004,704.0,1044.0
388,18/01/2005,873.0,1620.0
389,06/10/2004,865.0,1840.0
390,02/08/2004,854.0,1297.0


In [19]:
classSparkAirDat.minmax("`PT08.S1(CO)`", groupby = "`PT08.S2(NMHC)`")

,PT08.S2(NMHC),min(PT08.S1(CO)),max(PT08.S1(CO))
0,833.0,872.0,1112.0
1,496.0,832.0,883.0
2,471.0,691.0,790.0
3,1238.0,1088.0,1531.0
4,1088.0,1018.0,1408.0
...,...,...,...
1242,1217.0,1173.0,1520.0
1243,1138.0,1168.0,1276.0
1244,1413.0,1319.0,1560.0
1245,1476.0,1518.0,1626.0


The original data has $9357$ rows, so observe the `groupby` option works appropriately.

## Converted (from `pandas`) Class

Now we verify the methods demonstrated prior also apply to `spark` data frames that have been "converted" from `pandas` data frames.

In [20]:
pdAirDat = pd.read_csv("AirQualityUCI.csv", sep = ';')
pdAirDat.head()

,Date,Time,CO(GT),PT08.S1(CO),NMHC(GT),C6H6(GT),PT08.S2(NMHC),NOx(GT),PT08.S3(NOx),NO2(GT),PT08.S4(NO2),PT08.S5(O3),T,RH,AH,Unnamed: 15,Unnamed: 16
0,10/03/2004,18.00.00,"2,6",1360.0,150.0,"11,9",1046.0,166.0,1056.0,113.0,1692.0,1268.0,"13,6","48,9","0,7578",NaN,NaN
1,10/03/2004,19.00.00,2,1292.0,112.0,"9,4",955.0,103.0,1174.0,92.0,1559.0,972.0,"13,3","47,7","0,7255",NaN,NaN
2,10/03/2004,20.00.00,"2,2",1402.0,88.0,"9,0",939.0,131.0,1140.0,114.0,1555.0,1074.0,"11,9","54,0","0,7502",NaN,NaN
3,10/03/2004,21.00.00,"2,2",1376.0,80.0,"9,2",948.0,172.0,1092.0,122.0,1584.0,1203.0,"11,0","60,0","0,7867",NaN,NaN
4,10/03/2004,22.00.00,"1,6",1272.0,51.0,"6,5",836.0,131.0,1205.0,116.0,1490.0,1110.0,"11,2","59,6","0,7888",NaN,NaN


In [21]:
classConvertAirDat = proj2class.SparkDataCheck.convert_sdf(spark, pdAirDat)
classConvertAirDat.df.show()

+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------+-----------+
|      Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|Unnamed: 15|Unnamed: 16|
+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------+-----------+
|10/03/2004|18.00.00|   2,6|     1360.0|   150.0|    11,9|       1046.0|  166.0|      1056.0|  113.0|      1692.0|     1268.0|13,6|48,9|0,7578|        NaN|        NaN|
|10/03/2004|19.00.00|     2|     1292.0|   112.0|     9,4|        955.0|  103.0|      1174.0|   92.0|      1559.0|      972.0|13,3|47,7|0,7255|        NaN|        NaN|
|10/03/2004|20.00.00|   2,2|     1402.0|    88.0|     9,0|        939.0|  131.0|      1140.0|  114.0|      1555.0|     1074.0|11,9|54,0|0,7502|        NaN|     

Again, take note of the types of data contained in each column, which do differ from the previous class.

In [22]:
classConvertAirDat.df.dtypes

[('Date', 'string'),
 ('Time', 'string'),
 ('CO(GT)', 'string'),
 ('PT08.S1(CO)', 'double'),
 ('NMHC(GT)', 'double'),
 ('C6H6(GT)', 'string'),
 ('PT08.S2(NMHC)', 'double'),
 ('NOx(GT)', 'double'),
 ('PT08.S3(NOx)', 'double'),
 ('NO2(GT)', 'double'),
 ('PT08.S4(NO2)', 'double'),
 ('PT08.S5(O3)', 'double'),
 ('T', 'string'),
 ('RH', 'string'),
 ('AH', 'string'),
 ('Unnamed: 15', 'double'),
 ('Unnamed: 16', 'double')]

Now we repeat the previous method from the Read-In Class section:

In [23]:
classConvertAirDat.numrange("`PT08.S1(CO)`", 1000, 1250).show()

+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------+-----------+------+
|      Date|    Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|Unnamed: 15|Unnamed: 16|Result|
+----------+--------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+-----------+-----------+------+
|10/03/2004|18.00.00|   2,6|     1360.0|   150.0|    11,9|       1046.0|  166.0|      1056.0|  113.0|      1692.0|     1268.0|13,6|48,9|0,7578|        NaN|        NaN| false|
|10/03/2004|19.00.00|     2|     1292.0|   112.0|     9,4|        955.0|  103.0|      1174.0|   92.0|      1559.0|      972.0|13,3|47,7|0,7255|        NaN|        NaN| false|
|10/03/2004|20.00.00|   2,2|     1402.0|    88.0|     9,0|        939.0|  131.0|      1140.0|  114.0|      1555.0|     1074.0

# Part II

In this section we switch gears from examining air data to NFL data, utilizing `Spark SQL` and `pandas-on-Spark`. We begin with the latter.

## `pandas-on-Spark`

We first read in the `csv` containing the NFL data after importing the appropriate libraries and functions and examine the first $5$ rows.

In [24]:
import pandas as pd
import numpy as np
import pyspark.pandas as ps

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [25]:
pandaNFL = pd.read_csv("weekly_nfl_data.csv", delimiter = ",")
pysparkNFL = ps.from_pandas(pandaNFL)
pysparkNFL.head()

26/03/27 01:21:15 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


For reference, this massive data frame contains $128 873$ rows and $53$ columns, seen below.

In [26]:
pysparkNFL.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

We will now examine some statistics pertaining to the quarterbacks for the (regular) seasons $2005$ to $2023$ (inclusive). We begin by subsetting accordingly.

In [27]:
relData = pysparkNFL[(pysparkNFL["position"] == "QB") & (pysparkNFL["season_type"] == "REG") & (pysparkNFL["season"] >= 2005) & (pysparkNFL["season"] <= 2023)]
relData.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
29406,00-0000722,None,Tony Banks,QB,QB,None,HOU,2005,17,REG,SF,14,25,173.0,1,2.0,0.0,-0.0,0,0,0.0,0.0,7.0,-3.693779,0,0.0,0.032036,2,-2.0,0,0.0,0.0,0.0,0.000000,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,6.72,6.72
29426,00-0000865,None,Charlie Batch,QB,QB,None,PIT,2005,9,REG,GB,9,16,65.0,0,1.0,1.0,6.0,1,0,0.0,0.0,4.0,-6.609788,0,0.0,0.025249,8,14.0,0,0.0,0.0,1.0,-1.614163,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,2.00,2.00
29427,00-0000865,None,Charlie Batch,QB,QB,None,PIT,2005,10,REG,CLE,13,19,150.0,0,0.0,0.0,-0.0,0,0,0.0,0.0,7.0,5.193222,0,0.0,0.171917,3,16.0,1,0.0,0.0,2.0,0.737467,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,13.60,13.60
29428,00-0000865,None,Charlie Batch,QB,QB,None,PIT,2005,16,REG,CLE,1,1,31.0,1,0.0,0.0,-0.0,0,0,0.0,0.0,1.0,4.662244,0,0.0,NaN,0,0.0,0,0.0,0.0,0.0,NaN,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,5.24,5.24
29447,00-0001335,None,Jeff Blake,QB,QB,None,CHI,2005,2,REG,DET,1,1,11.0,0,0.0,0.0,-0.0,0,0,0.0,0.0,1.0,1.514100,0,0.0,NaN,1,-1.0,0,0.0,0.0,0.0,-2.570310,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,0.34,0.34


Next, we subset $8$ columns.

In [28]:
relDatSubset = relData[["player_display_name", "season", "week", "completions", "attempts", "passing_yards", "passing_tds", "interceptions"]]
relDatSubset.head()

,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
29406,Tony Banks,2005,17,14,25,173.0,1,2.0
29426,Charlie Batch,2005,9,9,16,65.0,0,1.0
29427,Charlie Batch,2005,10,13,19,150.0,0,0.0
29428,Charlie Batch,2005,16,1,1,31.0,1,0.0
29447,Jeff Blake,2005,2,1,1,11.0,0,0.0


Now we find the `sum` and `mean` for every player in the above table, grouped by season.

In [29]:
relDatSubsetMeans = relDatSubset.groupby(["player_display_name", "season"]).mean()
relDatSubsetMeans.head()

,,week,completions,attempts,passing_yards,passing_tds,interceptions
player_display_name,season,,,,,,
Jeff Blake,2005,9.500000,4.000000,4.500000,27.500000,0.500000,0.000000
Daunte Culpepper,2005,4.428571,19.857143,30.857143,223.428571,0.857143,1.714286
Kerry Collins,2005,8.933333,20.133333,37.733333,250.600000,1.333333,0.866667
Tony Banks,2005,17.000000,14.000000,25.000000,173.000000,1.000000,2.000000
Charlie Batch,2005,11.666667,7.666667,12.000000,82.000000,0.333333,0.333333


In [30]:
relDatSubsetSums = relDatSubset.groupby(["player_display_name", "season"]).sum()
relDatSubsetSums.head()

,,week,completions,attempts,passing_yards,passing_tds,interceptions
player_display_name,season,,,,,,
Jeff Blake,2005,19,8,9,55.0,1,0.0
Daunte Culpepper,2005,31,139,216,1564.0,6,12.0
Kerry Collins,2005,134,302,566,3759.0,20,13.0
Tony Banks,2005,17,14,25,173.0,1,2.0
Charlie Batch,2005,35,23,36,246.0,1,1.0


Now we compute a new column, comprised of the ratio of `completions` to `attempts`.

In [31]:
relDatSubsetSums["completion_percentage"] = relDatSubsetSums["completions"] / relDatSubsetSums["attempts"]
relDatSubsetSums.head()

,,week,completions,attempts,passing_yards,passing_tds,interceptions,completion_percentage
player_display_name,season,,,,,,,
Jeff Blake,2005,19,8,9,55.0,1,0.0,0.888889
Daunte Culpepper,2005,31,139,216,1564.0,6,12.0,0.643519
Kerry Collins,2005,134,302,566,3759.0,20,13.0,0.533569
Tony Banks,2005,17,14,25,173.0,1,2.0,0.560000
Charlie Batch,2005,35,23,36,246.0,1,1.0,0.638889


Then another column, comprised of the ratio of `passing_tds` to `interceptions`.

In [32]:
relDatSubsetSums["td_int_ratio"] = relDatSubsetSums["passing_tds"] / relDatSubsetSums["interceptions"]
relDatSubsetSums.head()

,,week,completions,attempts,passing_yards,passing_tds,interceptions,completion_percentage,td_int_ratio
player_display_name,season,,,,,,,,
Jeff Blake,2005,19,8,9,55.0,1,0.0,0.888889,inf
Daunte Culpepper,2005,31,139,216,1564.0,6,12.0,0.643519,0.500000
Kerry Collins,2005,134,302,566,3759.0,20,13.0,0.533569,1.538462
Tony Banks,2005,17,14,25,173.0,1,2.0,0.560000,0.500000
Charlie Batch,2005,35,23,36,246.0,1,1.0,0.638889,1.000000


Finally, we restrict the results to include those whose total attempts (by player in each season) exceeds $50$.

In [33]:
ResultUnsorted = relDatSubsetSums[relDatSubsetSums["attempts"] > 50]
ResultUnsorted.head()

,,week,completions,attempts,passing_yards,passing_tds,interceptions,completion_percentage,td_int_ratio
player_display_name,season,,,,,,,,
Daunte Culpepper,2005,31,139,216,1564.0,6,12.0,0.643519,0.500000
Kerry Collins,2005,134,302,566,3759.0,20,13.0,0.533569,1.538462
Jeff Garcia,2005,69,102,173,937.0,3,6.0,0.589595,0.500000
Brett Favre,2005,147,372,607,3881.0,20,29.0,0.612850,0.689655
Todd Bouman,2005,60,68,122,722.0,2,7.0,0.557377,0.285714


We then sort the rows, indicating the first $40$ values having the highest `completion_percentage`.

In [34]:
HighestCompletion = ResultUnsorted.sort_values(by = "completion_percentage", ascending = False)
HighestCompletion.head(40)

week  completions  attempts  passing_yards  passing_tds  interceptions  completion_percentage  td_int_ratio
player_display_name season                                                                                                             
C.J. Beathard       2023      65           40        53          349.0            1            0.0               0.754717           inf
Colt McCoy          2021      62           74        99          740.0            3            1.0               0.747475      3.000000
Matt Schaub         2019      52           50        67          580.0            3            1.0               0.746269      3.000000
Drew Brees          2018     130          364       489         3992.0           32            5.0               0.744376      6.400000
                    2019     119          281       378         2979.0           27            4.0               0.743386      6.750000
Mason Rudolph       2023      66           55        74          719.0            3            0.0               0.743243           inf
Taysom Hill         2020     147           88       121          928.0            4            2.0               0.727273      2.000000
Nick Foles          2018      51          141       195         1413.0            7            4.0               0.723077      1.750000
Drew Brees          2017     148          386       536         4334.0           23            8.0               0.720149      2.875000
Sam Bradford        2016     146          395       552         3877.0           20            5.0               0.715580      4.000000
Drew Brees          2011     142          471       660         5535.0           46           14.0               0.713636      3.285714
Colt McCoy          2014      57           91       128         1057.0            4            3.0               0.710938      1.333333
Aaron Rodgers       2020     148          372       526         4299.0           48            5.0               0.707224      9.600000
Bailey Zappe        2022      22           65        92          781.0            5            3.0               0.706522      1.666667
Drew Brees          2009     131          363       514         4388.0           34           11.0               0.706226      3.090909
                    2020      97          275       390         2942.0           24            6.0               0.705128      4.000000
Joe Burrow          2021     143          366       520         4611.0           34           14.0               0.703846      2.428571
Derek Carr          2019     147          361       513         4054.0           21            8.0               0.703704      2.625000
Jake Browning       2023     117          171       243         1936.0           12            7.0               0.703704      1.714286
Chase Daniel        2019      20           45        64          435.0            3            2.0               0.703125      1.500000
Ryan Tannehill      2019     128          201       286         2742.0           22            6.0               0.702797      3.666667
Deshaun Watson      2020     145          382       544         4823.0           33            7.0               0.702206      4.714286
Alex Smith          2012      63          153       218         1737.0           13            5.0               0.701835      2.600000
Kirk Cousins        2018     143          425       606         4298.0           30           10.0               0.701320      3.000000
Jamie Martin        2005      92          124       177         1277.0            5            7.0               0.700565      0.714286
Drew Brees          2016     148          471       673         5208.0           37           15.0               0.699851      2.466667
Tony Romo           2014     133          304       435         3705.0           34            9.0               0.698851      3.777778
Matt Ryan           2016     142          373       534         4944.0           38 

Finally, we sort them by `td_int_ratio`.

In [35]:
HighestTdIntRatio = ResultUnsorted.sort_values(by = "td_int_ratio", ascending = False)
HighestTdIntRatio.head(40)

,,week,completions,attempts,passing_yards,passing_tds,interceptions,completion_percentage,td_int_ratio
player_display_name,season,,,,,,,,
Matt Schaub,2005,65,33,64,495.0,4,0.0,0.515625,inf
Charlie Batch,2006,71,30,52,477.0,5,0.0,0.576923,inf
Todd Collins,2007,62,67,105,888.0,5,0.0,0.638095,inf
Troy Smith,2007,62,40,76,452.0,2,0.0,0.526316,inf
Jake Locker,2011,51,34,66,542.0,4,0.0,0.515152,inf
Derek Anderson,2014,30,65,97,701.0,5,0.0,0.670103,inf
Brian Hoyer,2016,27,134,200,1445.0,6,0.0,0.670000,inf
Nick Foles,2016,17,36,55,410.0,3,0.0,0.654545,inf
Jimmy Garoppolo,2016,49,43,63,502.0,4,0.0,0.682540,inf


## `Spark SQL` Dataframe

We repeat the previous steps, now utilizing the `Spark SQL` syntax. Note the existing `spark` session is still active and so we do not need to use the following code, again:

In [36]:
# from pyspark.sql import SparkSession
# spark = SparkSession.builder \
#         .appName("proj2") \
#         .config("spark.sql.ansi.enabled", "false") \
#         .getOrCreate()

We will read in the NFL data again, this time in a `spark` data frame. This will be much less visually "appealing" due to the number of columns, which we will shortly deal with.

In [37]:
sparkNFLdata = spark.read.load("weekly_nfl_data.csv", format = "csv", sep = ",", inferSchema = "true", header = "true")
sparkNFLdata.show(5)

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sack_

Again we list the (copious quantity of) columns.

In [38]:
sparkNFLdata.columns

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

Now we restrict the data to that containing only quarterbacks during the (regular) seasons $2005$ to $2023$.

In [39]:
sparkRelDat = sparkNFLdata.filter((sparkNFLdata.position == "QB") & (sparkNFLdata.season_type == "REG") & (sparkNFLdata.season >= 2005) & (sparkNFLdata.season <= 2023))
sparkRelDat.show(5)

+----------+-----------+-------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+-----------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name|player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sa

Finally (thankfully), we reduce the number of columns.

In [40]:
sparkRelDatSubset = sparkRelDat.select("player_display_name", "season", "week", "completions", "attempts", "passing_yards", "passing_tds", "interceptions")
sparkRelDatSubset.show(5)

+-------------------+------+----+-----------+--------+-------------+-----------+-------------+
|player_display_name|season|week|completions|attempts|passing_yards|passing_tds|interceptions|
+-------------------+------+----+-----------+--------+-------------+-----------+-------------+
|         Tony Banks|  2005|  17|         14|      25|        173.0|          1|          2.0|
|      Charlie Batch|  2005|   9|          9|      16|         65.0|          0|          1.0|
|      Charlie Batch|  2005|  10|         13|      19|        150.0|          0|          0.0|
|      Charlie Batch|  2005|  16|          1|       1|         31.0|          1|          0.0|
|         Jeff Blake|  2005|   2|          1|       1|         11.0|          0|          0.0|
+-------------------+------+----+-----------+--------+-------------+-----------+-------------+
only showing top 5 rows


(**SO** much better!) Now we find the (1) mean, and (2) sum across each player / season combination.

In [41]:
sparkRelDatSubsetMeans = sparkRelDatSubset.groupBy("player_display_name", "season").mean()
sparkRelDatSubsetMeans.show(5)

+-------------------+------+-----------+-----------------+------------------+------------------+------------------+------------------+------------------+
|player_display_name|season|avg(season)|        avg(week)|  avg(completions)|     avg(attempts)|avg(passing_yards)|  avg(passing_tds)|avg(interceptions)|
+-------------------+------+-----------+-----------------+------------------+------------------+------------------+------------------+------------------+
|      Jake Delhomme|  2006|     2006.0|7.615384615384615| 20.23076923076923| 33.15384615384615|215.76923076923077|1.3076923076923077|0.8461538461538461|
|       Jake Plummer|  2005|     2005.0|              9.0|           17.3125|              28.5|           210.375|             1.125|            0.4375|
|        Matt Schaub|  2006|     2006.0|             12.0|               3.6|               5.4|              41.6|               0.2|               0.4|
|        Vince Young|  2006|     2006.0|9.533333333333333|12.266666666666667

In [42]:
sparkRelDatSubsetSums = sparkRelDatSubset.groupBy("player_display_name", "season").sum()
sparkRelDatSubsetSums.show(5)

+-------------------+------+-----------+---------+----------------+-------------+------------------+----------------+------------------+
|player_display_name|season|sum(season)|sum(week)|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|
+-------------------+------+-----------+---------+----------------+-------------+------------------+----------------+------------------+
|      Jake Delhomme|  2006|      26078|       99|             263|          431|            2805.0|              17|              11.0|
|       Jake Plummer|  2005|      32080|      144|             277|          456|            3366.0|              18|               7.0|
|        Matt Schaub|  2006|      10030|       60|              18|           27|             208.0|               1|               2.0|
|        Vince Young|  2006|      30090|      143|             184|          356|            2199.0|              12|              13.0|
|      Kerry Collins|  2007|      12042| 

Now we append two new columns to our subsetted menu, for the `completion_percentage` and `td_int_ratio`. First, we change the `sum(completions)`, `sum(attempts)`, `sum(passing_yards)`, and `sum(interceptions)` columns to more "programming-friendly" names.

In [43]:
sparkRelDatSubsetSums = sparkRelDatSubsetSums.withColumnRenamed("sum(completions)", "sum_completions").withColumnRenamed("sum(attempts)", "sum_attempts") \
                                             .withColumnRenamed("sum(passing_yards)", "sum_passing_yards").withColumnRenamed("sum(interceptions)", "sum_interceptions")
sparkRelDatSubsetSums.show(5)

+-------------------+------+-----------+---------+---------------+------------+-----------------+----------------+-----------------+
|player_display_name|season|sum(season)|sum(week)|sum_completions|sum_attempts|sum_passing_yards|sum(passing_tds)|sum_interceptions|
+-------------------+------+-----------+---------+---------------+------------+-----------------+----------------+-----------------+
|      Jake Delhomme|  2006|      26078|       99|            263|         431|           2805.0|              17|             11.0|
|       Jake Plummer|  2005|      32080|      144|            277|         456|           3366.0|              18|              7.0|
|        Matt Schaub|  2006|      10030|       60|             18|          27|            208.0|               1|              2.0|
|        Vince Young|  2006|      30090|      143|            184|         356|           2199.0|              12|             13.0|
|      Kerry Collins|  2007|      12042|       48|             50|   

In [44]:
newSparkRelDatSubsetSums = sparkRelDatSubsetSums.withColumn('completion_percentage', sparkRelDatSubsetSums.sum_completions / sparkRelDatSubsetSums.sum_attempts)
newSparkRelDatSubsetSums.show(5)

+-------------------+------+-----------+---------+---------------+------------+-----------------+----------------+-----------------+---------------------+
|player_display_name|season|sum(season)|sum(week)|sum_completions|sum_attempts|sum_passing_yards|sum(passing_tds)|sum_interceptions|completion_percentage|
+-------------------+------+-----------+---------+---------------+------------+-----------------+----------------+-----------------+---------------------+
|      Jake Delhomme|  2006|      26078|       99|            263|         431|           2805.0|              17|             11.0|   0.6102088167053364|
|       Jake Plummer|  2005|      32080|      144|            277|         456|           3366.0|              18|              7.0|   0.6074561403508771|
|        Matt Schaub|  2006|      10030|       60|             18|          27|            208.0|               1|              2.0|   0.6666666666666666|
|        Vince Young|  2006|      30090|      143|            184|    

In [45]:
PUSparkRelDatSubsetSums = newSparkRelDatSubsetSums.withColumn('td_int_ratio', newSparkRelDatSubsetSums.sum_passing_yards / newSparkRelDatSubsetSums.sum_interceptions)
PUSparkRelDatSubsetSums.show(5)

+-------------------+------+-----------+---------+---------------+------------+-----------------+----------------+-----------------+---------------------+------------------+
|player_display_name|season|sum(season)|sum(week)|sum_completions|sum_attempts|sum_passing_yards|sum(passing_tds)|sum_interceptions|completion_percentage|      td_int_ratio|
+-------------------+------+-----------+---------+---------------+------------+-----------------+----------------+-----------------+---------------------+------------------+
|      Jake Delhomme|  2006|      26078|       99|            263|         431|           2805.0|              17|             11.0|   0.6102088167053364|             255.0|
|       Jake Plummer|  2005|      32080|      144|            277|         456|           3366.0|              18|              7.0|   0.6074561403508771|480.85714285714283|
|        Matt Schaub|  2006|      10030|       60|             18|          27|            208.0|               1|              2.

Notice that instead of division-by-zero being treated as $\infty$, in this setup the system (correctly, in my professional opinion) treats division-by-zero as `NULL`. As before, we subset the rows to only include player/season combinations where `sum_attempts` (no special characters, thank goodness!) is at least $50$.

In [46]:
sparkResultUnsorted = PUSparkRelDatSubsetSums.filter(PUSparkRelDatSubsetSums.sum_attempts > 50)
sparkResultUnsorted.show(5)

+-------------------+------+-----------+---------+---------------+------------+-----------------+----------------+-----------------+---------------------+------------------+
|player_display_name|season|sum(season)|sum(week)|sum_completions|sum_attempts|sum_passing_yards|sum(passing_tds)|sum_interceptions|completion_percentage|      td_int_ratio|
+-------------------+------+-----------+---------+---------------+------------+-----------------+----------------+-----------------+---------------------+------------------+
|      Jake Delhomme|  2006|      26078|       99|            263|         431|           2805.0|              17|             11.0|   0.6102088167053364|             255.0|
|       Jake Plummer|  2005|      32080|      144|            277|         456|           3366.0|              18|              7.0|   0.6074561403508771|480.85714285714283|
|        Vince Young|  2006|      30090|      143|            184|         356|           2199.0|              12|             13.

Now we sort the results (descending, naturally) by `completion_percentage`.

In [47]:
sparkHighestCompletion = sparkResultUnsorted.sort(sparkResultUnsorted.completion_percentage, ascending = False)
sparkHighestCompletion.show(10)

+-------------------+------+-----------+---------+---------------+------------+-----------------+----------------+-----------------+---------------------+------------+
|player_display_name|season|sum(season)|sum(week)|sum_completions|sum_attempts|sum_passing_yards|sum(passing_tds)|sum_interceptions|completion_percentage|td_int_ratio|
+-------------------+------+-----------+---------+---------------+------------+-----------------+----------------+-----------------+---------------------+------------+
|      C.J. Beathard|  2023|      12138|       65|             40|          53|            349.0|               1|              0.0|   0.7547169811320755|        NULL|
|         Colt McCoy|  2021|      14147|       62|             74|          99|            740.0|               3|              1.0|   0.7474747474747475|       740.0|
|        Matt Schaub|  2019|      10095|       52|             50|          67|            580.0|               3|              1.0|    0.746268656716418|      

And, lastly, by `td_int_ratio`.

In [48]:
sparkHighestTdIntRatio = sparkResultUnsorted.sort(sparkResultUnsorted.td_int_ratio, ascending = False)
sparkHighestTdIntRatio.show(10)

+-------------------+------+-----------+---------+---------------+------------+-----------------+----------------+-----------------+---------------------+------------+
|player_display_name|season|sum(season)|sum(week)|sum_completions|sum_attempts|sum_passing_yards|sum(passing_tds)|sum_interceptions|completion_percentage|td_int_ratio|
+-------------------+------+-----------+---------+---------------+------------+-----------------+----------------+-----------------+---------------------+------------+
|      Aaron Rodgers|  2018|      32288|      146|            372|         597|           4442.0|              25|              2.0|   0.6231155778894473|      2221.0|
|        Damon Huard|  2006|      20060|       69|            148|         244|           1878.0|              11|              1.0|   0.6065573770491803|      1878.0|
|        Josh McCown|  2013|      16104|       92|            149|         224|           1829.0|              13|              1.0|   0.6651785714285714|      

(Dr Post, I just want you to know that I don't "sportsball" very much, but for some unknown reason I do not like Brady and prefer Rodgers. I think it's because I like the Green Bay Packers because I like the idea of a City, rather than a Person, owning a Team.)